In [1]:
pip install pandas plotly matplotlib seaborn

Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd
import plotly.express as px 
import time
import psutil 
import os

In [2]:
#Loading Dataset
start_time = time.time()
df = pd.read_csv('fire_archive_J1V-C2_726206.csv')
end_time = time.time()
print(f"Time taken to load dataset: {end_time - start_time} seconds")
df.info()

Time taken to load dataset: 10.24129319190979 seconds
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22620926 entries, 0 to 22620925
Data columns (total 15 columns):
 #   Column      Dtype  
---  ------      -----  
 0   latitude    float64
 1   longitude   float64
 2   brightness  float64
 3   scan        float64
 4   track       float64
 5   acq_date    object 
 6   acq_time    int64  
 7   satellite   object 
 8   instrument  object 
 9   confidence  object 
 10  version     int64  
 11  bright_t31  float64
 12  frp         float64
 13  daynight    object 
 14  type        int64  
dtypes: float64(7), int64(3), object(5)
memory usage: 2.5+ GB


In [3]:
df.head()

,latitude,longitude,brightness,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_t31,frp,daynight,type
0,70.94708,74.53897,331.97,0.54,0.51,2024-02-01,7,N20,VIIRS,n,2,246.54,3.93,N,0
1,68.41541,83.62364,367.00,0.50,0.66,2024-02-01,7,N20,VIIRS,h,2,249.76,7.27,N,2
2,66.66642,80.44739,367.00,0.59,0.70,2024-02-01,7,N20,VIIRS,h,2,249.78,13.21,N,2
3,70.99835,73.84378,345.58,0.53,0.50,2024-02-01,7,N20,VIIRS,n,2,252.52,23.67,N,0
4,67.28266,83.03193,329.51,0.58,0.70,2024-02-01,7,N20,VIIRS,n,2,251.59,3.59,N,2


In [ ]:
#Cleaning Dataset
df.shape

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

Checking processing time for the aggregation functions

In [3]:
def assign_period(month):
    if month in [2,3]:
        return "Feb–Mar"
    elif month in [4,5]:
        return "Apr–May"
    elif month in [6,7]:
        return "Jun–Jul"
    elif month in [8,9]:
        return "Aug–Sep"
    elif month in [10,11]:
        return "Oct–Nov"
    else:
        return "Dec–Jan"

process_start = time.time()
df['acq_date'] = pd.to_datetime(df['acq_date'])
df['month'] = df['acq_date'].dt.month
df['period'] = df['month'].apply(assign_period)

fires_monthly = df.groupby(df['acq_date'].dt.to_period("M")).size()
process_time = time.time() - process_start

print(f"Processing Time (Aggregations Only): {process_time:.2f} seconds")

Processing Time (Aggregations Only): 3.08 seconds


#Creating Visualizations

In [ ]:
#Creating Sample dataset

df['acq_date'] = pd.to_datetime(df['acq_date'])
df['month'] = df['acq_date']. dt.month

def assign_period(month):
    if month in [2,3]:
        return "Feb–Mar"
    elif month in [4,5]:
        return "Apr–May"
    elif month in [6,7]:
        return "Jun–Jul"
    elif month in [8,9]:
        return "Aug–Sep"
    elif month in [10,11]:
        return "Oct–Nov"
    else:
        return "Dec–Jan"

df['period'] = df['month'].apply(assign_period)

df_sample = df.sample(n=40000, random_state=42)
df_sample.shape


(40000, 17)

Fire Density Heat Maps

In [16]:
import plotly.express as px

for period in df_sample['period'].unique():

    subset = df_sample[df_sample['period'] == period]

    fig = px.density_mapbox(
        subset,
        lat="latitude",
        lon="longitude",
        radius=3,
        zoom=1,
        mapbox_style="carto-positron",
        title=f"Wildfire Density ({period})"
    )

    fig.show()

/var/folders/7y/fhg43h095q7blcbrnc479ly00000gn/T/ipykernel_49056/1494433411.py:7: DeprecationWarning: *density_mapbox* is deprecated! Use *density_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig = px.density_mapbox(


/var/folders/7y/fhg43h095q7blcbrnc479ly00000gn/T/ipykernel_49056/1494433411.py:7: DeprecationWarning: *density_mapbox* is deprecated! Use *density_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig = px.density_mapbox(


/var/folders/7y/fhg43h095q7blcbrnc479ly00000gn/T/ipykernel_49056/1494433411.py:7: DeprecationWarning: *density_mapbox* is deprecated! Use *density_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig = px.density_mapbox(


/var/folders/7y/fhg43h095q7blcbrnc479ly00000gn/T/ipykernel_49056/1494433411.py:7: DeprecationWarning: *density_mapbox* is deprecated! Use *density_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig = px.density_mapbox(


/var/folders/7y/fhg43h095q7blcbrnc479ly00000gn/T/ipykernel_49056/1494433411.py:7: DeprecationWarning: *density_mapbox* is deprecated! Use *density_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig = px.density_mapbox(


/var/folders/7y/fhg43h095q7blcbrnc479ly00000gn/T/ipykernel_49056/1494433411.py:7: DeprecationWarning: *density_mapbox* is deprecated! Use *density_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig = px.density_mapbox(


Global Fire Hotspots

In [17]:
for period in df_sample['period'].unique():

    subset = df_sample[df_sample['period'] == period]

    fig = px.scatter_geo(
        subset,
        lat="latitude",
        lon="longitude",
        opacity=0.5,
        title=f"Wildfire Hotspots ({period})"
    )

    fig.show()

Fire Detections Over Time

In [5]:
fires_monthly = df.groupby(df['acq_date'].dt.to_period("M")).size().reset_index(name="fire_count")

fires_monthly['acq_date'] = fires_monthly['acq_date'].astype(str)

time_taken = time.time() - process_time
print(f"Time taken for data processing and visualization: {time_taken} seconds")

Time taken for data processing and visualization: 13.361721992492676 seconds


In [23]:
fig = px.line(
    fires_monthly,
    x="acq_date",
    y="fire_count",
    title="Wildfire Detections Over Time",
    labels = {
        'acq_date': 'Acquired Date',
        'fire_count': 'Fire Count'
    }
)

fig.show()

FRP Map Sizing

In [25]:
import plotly.express as px

for period in df_sample['period'].unique():

    subset = df_sample[df_sample['period'] == period]

    fig = px.scatter_geo(
        subset,
        lat="latitude",
        lon="longitude",
        size="frp",
        color="frp",
        color_continuous_scale="YlOrRd",
        opacity=0.6,
        title=f"Wildfire Intensity by FRP ({period})",
        labels={
            "frp": "Fire Radiative Power",
            "latitude": "Latitude",
            "longitude": "Longitude"
        }
    )

    fig.show()